In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver Layer: Customer SCD Type 2 Dimension (`silver.dim_customers_scd2`)
# MAGIC 
# MAGIC - **Sources**: `bronze.user_addr_scd2`, `bronze.user`
# MAGIC - **Target**: `silver.dim_customers_scd2` (Unity Catalog managed Delta table)
# MAGIC - **Key Mechanics**:
# MAGIC   - Tracks address changes chronologically using window ordering.
# MAGIC   - Computes validity dates (`valid_from`, `valid_to`) and `is_current`.
# MAGIC   - Generates deterministic surrogate key `customer_sk` via SHA-256.

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Ensure silver schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# COMMAND ----------

# 2. Read Bronze address history and user profile data
df_addr = spark.table("bronze.user_addr_scd2")
df_users = spark.table("bronze.user") if spark.catalog.tableExists("bronze.user") else None

# COMMAND ----------


In [0]:

# 3. Build SCD Type 2 date ranges per customer
window_spec = Window.partitionBy("usr_id").orderBy(F.col("usr_addr_change_date").asc())

df_scd2_prep = (
    df_addr
    .filter(F.col("usr_id").isNotNull() & F.col("usr_addr_change_date").isNotNull())
    .dropDuplicates(["usr_id", "usr_addr_change_date"])
    .withColumn("valid_from", F.to_date(F.col("usr_addr_change_date")))
    .withColumn(
        "next_change_date",
        F.lead("usr_addr_change_date").over(window_spec)
    )
    .withColumn(
        "valid_to",
        F.coalesce(
            F.date_sub(F.to_date(F.col("next_change_date")), 1),
            F.to_date(F.lit("9999-12-31"))
        )
    )
    .withColumn(
        "is_current",
        F.when(F.col("valid_to") == F.to_date(F.lit("9999-12-31")), F.lit(True)).otherwise(F.lit(False))
    )
    .drop("next_change_date")
)


In [0]:

# COMMAND ----------

# 4. Generate Surrogate Key (SHA-256 of natural key + valid_from)
df_scd2_with_sk = df_scd2_prep.withColumn(
    "customer_sk",
    F.sha2(
        F.concat_ws("||", F.col("usr_id").cast("string"), F.col("valid_from").cast("string")),
        256
    )
)

# COMMAND ----------


In [0]:

# 5. Enrich with profile metadata (avoiding overlapping column collisions)
if df_users is not None:
    audit_cols = ["_ingested_at", "_source_file", "_batch_id", "_row_hash"]
    overlapping_cols = [c for c in df_users.columns if c in df_scd2_with_sk.columns and c != "usr_id"]
    cols_to_drop = list(set(audit_cols + overlapping_cols))
    
    df_users_clean = df_users.drop(*cols_to_drop).dropDuplicates(["usr_id"])
    
    df_dim_customers = (
        df_scd2_with_sk.join(df_users_clean, on="usr_id", how="left")
        .withColumn("_transformed_at", F.current_timestamp())
    )
else:
    df_dim_customers = df_scd2_with_sk.withColumn("_transformed_at", F.current_timestamp())



In [0]:
# COMMAND ----------

# 6. Save as managed Delta table in Unity Catalog & Optimize
(
    df_dim_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_customers_scd2")
)

spark.sql("OPTIMIZE silver.dim_customers_scd2 ZORDER BY (usr_id, is_current)")

# COMMAND ----------

# 7. Validate SCD2 results
display(
    spark.table("silver.dim_customers_scd2")
    .select("customer_sk", "usr_id", "usr_city", "usr_state", "valid_from", "valid_to", "is_current")
    .orderBy("usr_id", "valid_from")
    .limit(10)
)